#Task 2 Approach 2
# Using fine-tuned BERT approach

In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e0f524133187e3ad9ec630ae98bde76df7d16db94fea2497d3f37dee7f8deea2
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [ ]:
import pandas as pd
import gc

# Mount Google Drive (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

# File paths
train_path = "/content/drive/My Drive/ANLE/Assignment Datasets/propaganda_train.tsv"
val_path = "/content/drive/My Drive/ANLE/Assignment Datasets/propaganda_val.tsv"

# Read the TSV files into DataFrames
train_df = pd.read_csv(train_path, sep='\t')
val_df = pd.read_csv(val_path, sep='\t')

train_df.head()

Mounted at /content/drive


,label,tagged_in_context
0,not_propaganda,"No, <BOS> he <EOS> will not be confirmed."
1,not_propaganda,This declassification effort <BOS> won’t make ...
2,flag_waving,The Obama administration misled the <BOS> Amer...
3,not_propaganda,“It looks like we’re capturing the demise of t...
4,not_propaganda,"<BOS> Location: Westerville, Ohio <EOS>"


## Resample datasets

In [ ]:
import numpy as np

def resample_non_propaganda(df, keep_prob=0.2, seed=42):
    """
    Randomly drops non-propaganda instances from the DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame with 'label' column.
        keep_prob (float): Probability of keeping a non-propaganda instance.
        seed (int): Random seed for reproducibility.

    Returns:
        pd.DataFrame: A resampled DataFrame.
    """
    np.random.seed(seed)

    # Split into propaganda and non-propaganda
    is_non_propaganda = df['label'].str.strip() == "not_propaganda"

    # Keep all propaganda instances
    propaganda_df = df[~is_non_propaganda]

    # Randomly keep some non-propaganda instances
    non_propaganda_df = df[is_non_propaganda]
    keep_mask = np.random.rand(len(non_propaganda_df)) < keep_prob
    kept_non_propaganda = non_propaganda_df[keep_mask]

    # Combine back
    resampled_df = pd.concat([propaganda_df, kept_non_propaganda]).reset_index(drop=True)

    return resampled_df

train_df = resample_non_propaganda(train_df, keep_prob=0.8)
val_df = resample_non_propaganda(val_df, keep_prob=0.8)

## Preprocess Data and generate BIO tags

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
nltk.download('punkt')

# Function to extract tokens and BIO tags from a row containing <BOS> and <EOS> markers
def extract_tokens_and_bio_tags(row):
    """
    Parses a sentence containing <BOS> and <EOS> to extract tokens and BIO tags.

    Args:
        row (dict or pd.Series): A row with fields:
            - 'tagged_in_context': the sentence with <BOS>...</EOS> marking a labeled span
            - 'label': the label/category of the span (e.g., propaganda)

    Returns:
        tokens: Tokenized version of the sentence
        tags: Corresponding BIO tags for each token
    """
    sentence = row['tagged_in_context']
    label = row['label'].strip()  # Clean label string (e.g., 'propaganda')

    # Search for the span between <BOS> and <EOS>
    span_match = re.search(r"<BOS>(.*?)<EOS>", sentence)

    if not span_match:
        # No span found — treat entire sentence as 'O' (outside any entity)
        tokens = word_tokenize(sentence.replace("<BOS>", "").replace("<EOS>", ""))
        return tokens, ['O'] * len(tokens)

    # If a span is found, extract it and tokenize it
    span = span_match.group(1).strip()
    span_tokens = word_tokenize(span)

    # Remove <BOS>/<EOS> tags and tokenize the entire cleaned sentence
    clean_sentence = sentence.replace("<BOS>", "").replace("<EOS>", "")
    tokens = word_tokenize(clean_sentence)

    # Initialize all tokens as 'O' (non-entity)
    tags = ['O'] * len(tokens)

    # Look for exact token match of span_tokens in the tokenized sentence
    for i in range(len(tokens) - len(span_tokens) + 1):
        if tokens[i:i+len(span_tokens)] == span_tokens:
            # Tag beginning of span
            tags[i] = f"B-{label}"
            # Tag inside of span (if span is more than one token)
            for j in range(1, len(span_tokens)):
                tags[i + j] = f"I-{label}"
            break  # Stop after first correct match

    return tokens, tags

# Function to remove <BOS> and <EOS> tags from the sentence text
def clean_text(text):
    """
    Cleans a sentence by removing <BOS> and <EOS> markers.

    Args:
        text (str): Sentence possibly containing <BOS> and <EOS> tags.

    Returns:
        str: Cleaned sentence.
    """
    return text.replace('<BOS>', '').replace('<EOS>', '').strip()

# Function to assign full-span 'not_propaganda' tags to non-propaganda sentences
def reassign_non_propaganda_spans(row):
    """
    Converts all tokens in 'not_propaganda' labeled sentences into a single full-span entity,
    while preserving original spans for 'propaganda'.

    Args:
        row (dict or pd.Series): Must contain:
            - 'tokenised_sents': tokenized sentence
            - 'label': label for the span/sentence
            - 'BIO_tags': original BIO tags (used if label is propaganda)

    Returns:
        tokens : tokenized sentence and BIO tag sequence
    """
    tokens = row['tokenised_sents']
    label = row['label'].strip()

    if label == "not_propaganda":
        # Treat entire sentence as one long 'not_propaganda' entity
        tags = ['B-not_propaganda'] + ['I-not_propaganda'] * (len(tokens) - 1)
        return tokens, tags
    else:
        # Keep original tagging for 'propaganda' examples
        return row['tokenised_sents'], row['BIO_tags']

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
# Apply the function to both train and val DataFrames
train_tokens_and_tags = train_df.apply(extract_tokens_and_bio_tags, axis=1)
val_tokens_and_tags = val_df.apply(extract_tokens_and_bio_tags, axis=1)

# Split tuple columns into two new columns
train_df[['tokenised_sents', 'BIO_tags']] = pd.DataFrame(train_tokens_and_tags.tolist(), index=train_df.index)
val_df[['tokenised_sents', 'BIO_tags']] = pd.DataFrame(val_tokens_and_tags.tolist(), index=val_df.index)

# Apply the function to reformat all non-propaganda tags
train_tokens_and_tags = train_df.apply(reassign_non_propaganda_spans, axis=1)
val_tokens_and_tags = val_df.apply(reassign_non_propaganda_spans, axis=1)

train_df[['tokenised_sents', 'BIO_tags']] = pd.DataFrame(train_tokens_and_tags.tolist(), index=train_df.index)
val_df[['tokenised_sents', 'BIO_tags']] = pd.DataFrame(val_tokens_and_tags.tolist(), index=val_df.index)

train_df['sents'] = train_df['tagged_in_context'].apply(clean_text)

In [ ]:
# Rebuild train_df
train_df = pd.DataFrame({
    'sents': train_df['tagged_in_context'].apply(clean_text),
    'tokenised_sents': train_df['tokenised_sents'],
    'BIO_tags': train_df['BIO_tags']
})

# Same for val_df
val_df = pd.DataFrame({
    'sents': val_df['tagged_in_context'].apply(clean_text),
    'tokenised_sents': val_df['tokenised_sents'],
    'BIO_tags': val_df['BIO_tags']
})

val_df.head()

,sents,tokenised_sents,BIO_tags
0,Mostly because the country would not last lon...,"[Mostly, because, the, country, would, not, la...","[O, O, B-causal_oversimplification, I-causal_o..."
1,Lyndon Johnson gets Earl Warren and Sen. Rich...,"[Lyndon, Johnson, gets, Earl, Warren, and, Sen...","[O, O, B-appeal_to_fear_prejudice, I-appeal_to..."
2,It must be exacted from him directly in order ...,"[It, must, be, exacted, from, him, directly, i...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,Is it any wonder that priests and laity alike ...,"[Is, it, any, wonder, that, priests, and, lait...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-n..."
4,Health workers have been asked to work with co...,"[Health, workers, have, been, asked, to, work,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


### Create Label Mappings and Convert BIO Tags to ID Sequences

In [ ]:
# Collect all unique BIO tags from train and val
all_tags = set(tag for tags in train_df['BIO_tags'].tolist() + val_df['BIO_tags'].tolist() for tag in tags)

# Ensure "O" comes first
all_tags.discard('O')  # Remove 'O' temporarily
all_tags = sorted(all_tags)  # Sort all other tags alphabetically
all_tags = ['O'] + all_tags   # Add 'O' at the beginning

# Create mapping dictionaries
label2id = {label: idx for idx, label in enumerate(all_tags)}
id2label = {idx: label for label, idx in label2id.items()}

# Encode labels in DataFrame
train_df['label_ids'] = train_df['BIO_tags'].apply(lambda tags: [label2id[tag] for tag in tags])
val_df['label_ids'] = val_df['BIO_tags'].apply(lambda tags: [label2id[tag] for tag in tags])

print(label2id)

{'O': 0, 'B-appeal_to_fear_prejudice': 1, 'B-causal_oversimplification': 2, 'B-doubt': 3, 'B-exaggeration,minimisation': 4, 'B-flag_waving': 5, 'B-loaded_language': 6, 'B-name_calling,labeling': 7, 'B-not_propaganda': 8, 'B-repetition': 9, 'I-appeal_to_fear_prejudice': 10, 'I-causal_oversimplification': 11, 'I-doubt': 12, 'I-exaggeration,minimisation': 13, 'I-flag_waving': 14, 'I-loaded_language': 15, 'I-name_calling,labeling': 16, 'I-not_propaganda': 17, 'I-repetition': 18}


## Introduce BERT tokenizer and model

In [ ]:
from transformers import AlbertForTokenClassification, AlbertTokenizerFast

tokenizer = AlbertTokenizerFast.from_pretrained("albert-base-v2")
model = AlbertForTokenClassification.from_pretrained("albert-base-v2", num_labels=len(label2id))

Some weights of AlbertForTokenClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Because ALBERT tokenizer breaks down rare words it has to be tagged accordingly by introducing special token that will be ignored

In [ ]:
def tokenize_and_align_labels_single(tokens, bio_labels, tokenizer, label2id):
    tokenized_inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        padding=False,
        truncation=True,
        return_offsets_mapping=True
    )

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)  # Special tokens like [CLS], [SEP]
        elif word_idx != previous_word_idx:
            label_ids.append(label2id[bio_labels[word_idx]])  # First subword keeps label
        else:
            label_ids.append(-100)  # Ignore subsequent subwords

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

In [ ]:
# For training
train_inputs_list = []

for idx, row in train_df.iterrows():
    inputs = tokenize_and_align_labels_single(row['tokenised_sents'], row['BIO_tags'], tokenizer, label2id)
    train_inputs_list.append(inputs)

# For validation
val_inputs_list = []

for idx, row in val_df.iterrows():
    inputs = tokenize_and_align_labels_single(row['tokenised_sents'], row['BIO_tags'], tokenizer, label2id)
    val_inputs_list.append(inputs)

def collate_inputs(input_list):
    keys = input_list[0].keys()
    batched = {k: [ex[k] for ex in input_list] for k in keys}
    return batched

train_inputs = collate_inputs(train_inputs_list)
val_inputs = collate_inputs(val_inputs_list)

#### Check that labels are aligned and correct

In [ ]:
print("Tokens:", tokenizer.convert_ids_to_tokens(train_inputs['input_ids'][5]))
print("Labels:", train_inputs['labels'][5])

Tokens: ['[CLS]', '▁trump', '▁has', '▁expelled', '▁more', '▁russian', '▁diplomat', 's', '▁and', '▁', 'sanctioned', '▁more', '▁russian', '▁', 'oli', 'gar', 'ch', 's', '▁than', '▁[', '▁obama', '▁', ']', '▁has', '▁', '.', '[SEP]']
Labels: [-100, 0, 0, 0, 9, 0, 0, -100, 0, 0, -100, 0, 0, 0, -100, -100, -100, -100, 0, 0, 0, 0, -100, 0, 0, -100, -100]


In [ ]:
import torch
from torch.utils.data import Dataset

class PropagandaDataset(Dataset):
    def __init__(self, inputs):
        self.input_ids = inputs['input_ids']
        self.attention_mask = inputs['attention_mask']
        self.labels = inputs['labels']

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[idx], dtype=torch.long),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Create datasets
train_dataset = PropagandaDataset(train_inputs)
val_dataset = PropagandaDataset(val_inputs)

##Load ALBERT Model for Token Classification

In [ ]:
from transformers import AlbertForTokenClassification

model = AlbertForTokenClassification.from_pretrained(
    "albert-base-v2",
    num_labels=len(label2id)  # Number of unique BIO labels
)

Some weights of AlbertForTokenClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Define a Function for Evaluation

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def compute_metrics(p):
    """
    Evaluate model generated sequence with seqeval
    """
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_predictions = []
    true_labels = []

    for pred, label in zip(predictions, labels):
        temp_pred = []
        temp_label = []
        for p_, l_ in zip(pred, label):
            if l_ != -100:  # Ignore padding tokens
                temp_pred.append(id2label[p_])
                temp_label.append(id2label[l_])
        true_predictions.append(temp_pred)
        true_labels.append(temp_label)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

## Set Training Arguments and Define Trainer
##### Code is taken from Lab9 b

In [ ]:
from transformers import TrainingArguments
from transformers import Trainer
import os
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)
os.environ["WANDB_DISABLED"] = "true"

#model = AlbertForTokenClassification.from_pretrained(
#    "albert-base-v2",
#    num_labels=len(label2id)  # Number of unique BIO labels
#)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",    # Evaluate every epoch
    learning_rate=4e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.2,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-124-60c9f1354a57>:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,2.731661,0.166304,0.296512,0.213092
2,0.310600,2.418103,0.224274,0.329457,0.266876
3,0.310600,2.223973,0.185145,0.333333,0.238062
4,0.259600,2.525827,0.178238,0.333333,0.232275
5,0.259600,2.744344,0.251101,0.331395,0.285714
6,0.150000,3.125767,0.252836,0.302326,0.275375
7,0.150000,3.056913,0.273141,0.348837,0.306383


In [ ]:
from seqeval.metrics import classification_report

# Predict
outputs = trainer.predict(val_dataset)
predictions = outputs.predictions.argmax(-1)

# Prepare Gold and Predicted Labels (Ignoring Padding)
true_labels = []
pred_labels = []

for i in range(len(val_dataset)):
    tokens = tokenizer.convert_ids_to_tokens(val_dataset[i]['input_ids'])
    labels = val_dataset[i]['labels']
    preds = predictions[i]

    true_sent = []
    pred_sent = []

    for p_, l_ in zip(preds, labels):
        if l_.item() != -100:  # Ignore padding
            true_sent.append(id2label[l_.item()])
            pred_sent.append(id2label[p_.item()])

    true_labels.append(true_sent)
    pred_labels.append(pred_sent)

# Print 5 random predictions and their tokens and gold labels
for idx in range(5):
    tokens = tokenizer.convert_ids_to_tokens(val_dataset[idx]['input_ids'])
    labels = val_dataset[idx]['labels']
    preds = predictions[idx]

    print("Tokens:", tokens)
    print("Gold Labels:", [id2label[id_.item()] for id_ in labels if id_.item() != -100])
    print("Predicted:", [id2label[pred.item()] for pred, label in zip(preds, labels) if label.item() != -100])
    print("-" * 50)

# Print Full Per-Class Metrics
print("\n--- Classification Report (Per-Class Precision, Recall, F1) ---")
print(classification_report(true_labels, pred_labels, digits=4))

Tokens: ['[CLS]', '▁mostly', '▁because', '▁the', '▁country', '▁would', '▁not', '▁last', '▁long', '▁without', '▁an', '▁outside', '▁high', '▁', 'iq', '▁elite', '▁to', '▁run', '▁the', '▁country', '▁', '.', '[SEP]']
Gold Labels: ['O', 'O', 'B-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'I-causal_oversimplification', 'O']
Predicted: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-exaggeration,minimisation', 'I-exaggeration,minimisation', 'O', 'O']
--------------------------------------------------
Tokens: ['[CLS]', '▁lyndon', '▁johnson', '▁gets', '▁e